# Phase 1 — Exploratory Data Analysis

## Overview
This notebook performs a complete exploration of the 7 macroeconomic time series downloaded from FRED (Federal Reserve Economic Data). The goals are:

1. **Understand the distribution** of US CPI inflation across 35 years (1990–2025)
2. **Identify autocorrelation structure** — how strongly does past inflation predict future inflation?
3. **Measure correlations** between macro indicators — which variables co-move with CPI?
4. **Detect seasonality and trend** via seasonal decomposition
5. **Flag data quality issues** before feature engineering

### Key finding preview
CPI inflation exhibits strong **positive autocorrelation** at lags 1–12 months (ACF plot), justifying our use of lag features. The PPI (Producer Price Index) and Fed Funds Rate show the highest correlation with inflation, consistent with cost-push and monetary theory.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Load Raw Data

We download 7 FRED series via `src/ingest.py`. All series are stored as monthly CSVs in `data/raw/`. GDP is quarterly and will be forward-filled to monthly frequency in Phase 2.

| Series | FRED ID | Frequency |
|--------|---------|-----------|
| CPI (All Urban, NSA) | CPIAUCNS | Monthly |
| M2 Money Supply | M2SL | Monthly |
| Unemployment Rate | UNRATE | Monthly |
| Fed Funds Rate | FEDFUNDS | Monthly |
| Real GDP Growth | A191RL1Q225SBEA | Quarterly |
| WTI Crude Oil | DCOILWTICO | Monthly |
| PPI (All Commodities) | PPIACO | Monthly |

In [ ]:
from src.ingest import download_all, load_raw

# Download if not already cached
# raw = download_all()  # Un-comment to fetch from FRED
raw = load_raw()

## 2. Summary Statistics

Key observations to note:
- **CPI** ranges from ~130 to ~315 over the period — a 2.4× increase over 35 years
- **Inflation rate** (YoY CPI %) has a mean around 2.7% with significant right-tail skew from COVID-era spikes
- **Oil price** has the highest coefficient of variation — it is the most volatile series

In [ ]:
# Build a combined DataFrame
df_raw = pd.DataFrame(raw)
df_raw['inflation_rate'] = df_raw['cpi'].pct_change(12) * 100
df_raw = df_raw.dropna()
print(df_raw.shape)
df_raw.head()

## 3. Inflation Rate Over Time

The chart reveals three distinct inflation regimes:
- **1990–2009**: Moderate inflation 1–5%, with brief spikes around Gulf War (1990) and commodity cycle (2007–2008)
- **2010–2019**: Persistently low inflation 0–2.5% — the "missing inflation" decade despite ultra-loose monetary policy
- **2020–2023**: COVID shock — supply disruption + fiscal stimulus drove inflation to a 40-year high of ~9% (June 2022)
- **2023–2025**: Rapid disinflation as Fed raised rates 525bps

This structural variation is why a **COVID regime dummy** is critical for our models.

## 4. Distribution and Normality

The Q-Q plot checks whether the inflation distribution is Gaussian. We expect departures in the tails — confirmed by the COVID-era outliers at the upper end. This motivates:
- Using **tree-based models** (XGBoost) that do not assume normality
- Keeping the ARIMA's differencing step to achieve stationarity

## 5. Correlation Matrix

**Interpretation guide:**
- Strong positive correlation between **PPI and inflation** (~0.8) confirms cost-push dynamics — rising producer costs pass through to consumer prices
- **M2 money supply** shows moderate positive correlation — consistent with monetarist theory, but with a multi-month lag (captured by lag features)
- **Unemployment** is negatively correlated with inflation — the classic Phillips Curve relationship (weaker post-2010)
- High correlation between CPI-level and M2-level (>0.95) signals multicollinearity — we use rate-of-change features instead of levels

## 6. Autocorrelation Analysis (ACF / PACF)

The **ACF (Autocorrelation Function)** shows how correlated inflation is with its own past values:
- Significant autocorrelation up to lag 12–24 confirms inflation is **highly persistent**
- This directly justifies our lag features: CPI(t-1), CPI(t-3), CPI(t-6), CPI(t-12)

The **PACF (Partial ACF)** isolates the direct effect at each lag (controlling for intermediate lags):
- Strong spike at lag 1, smaller at lags 2–3 → AR(1) or AR(2) structure in ARIMA
- This guides `auto_arima`'s parameter selection in Phase 3

## 7. Seasonal Decomposition

Seasonal decomposition splits the series into:
- **Trend**: the long-run direction (rising to 2022, falling since)
- **Seasonal**: recurring monthly patterns — CPI has mild seasonality (energy prices in winter, apparel in spring)
- **Residual**: unexplained variation — spikes here indicate structural breaks (2008 financial crisis, 2020 COVID shock)

The seasonal component supports including **month-of-year dummy variables** in our feature matrix.

In [ ]:
# Summary statistics
df_raw.describe().T

In [ ]:
# Plot inflation rate over time
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df_raw.index, df_raw['inflation_rate'], color='steelblue')
ax.axhline(0, color='red', linestyle='--', alpha=0.5)
ax.set_title('US Inflation Rate (YoY CPI Change)')
ax.set_ylabel('%')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of inflation
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df_raw['inflation_rate'], kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Inflation Rate Distribution')
from scipy import stats
stats.probplot(df_raw['inflation_rate'].dropna(), plot=axes[1])
axes[1].set_title('Q-Q Plot')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
corr_cols = ['inflation_rate', 'cpi', 'm2', 'unemployment', 'fed_funds', 'oil_wti', 'ppi', 'gdp_growth']
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df_raw[corr_cols].corr(), annot=True, fmt='.2f', cmap='RdYlGn', ax=ax, center=0)
ax.set_title('Correlation Matrix — Macro Indicators')
plt.tight_layout()
plt.show()

In [ ]:
# ACF / PACF for CPI inflation
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(df_raw['inflation_rate'].dropna(), lags=36, ax=axes[0])
axes[0].set_title('ACF — Inflation Rate')
plot_pacf(df_raw['inflation_rate'].dropna(), lags=36, ax=axes[1], method='ywm')
axes[1].set_title('PACF — Inflation Rate')
plt.tight_layout()
plt.show()

In [ ]:
# Seasonal decomposition
decomp = seasonal_decompose(df_raw['inflation_rate'].dropna(), model='additive', period=12)
fig = decomp.plot()
fig.set_size_inches(14, 8)
plt.suptitle('Seasonal Decomposition of CPI Inflation', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# All series side-by-side
plot_cols = ['m2', 'unemployment', 'fed_funds', 'oil_wti', 'ppi', 'gdp_growth']
fig, axes = plt.subplots(3, 2, figsize=(14, 12))
for ax, col in zip(axes.flatten(), plot_cols):
    ax.plot(df_raw.index, df_raw[col])
    ax.set_title(col)
    ax.grid(True, alpha=0.3)
plt.suptitle('Macro Indicators 1990–2025', y=1.01)
plt.tight_layout()
plt.show()